In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler


@dataclass(frozen=True)
class Paths:
    data: Path = Path("data") / "data.xlsx"
    output_dir: Path = Path("outputs") / "figure_8"


GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}
GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}
GASOLINE_ROOTS = GASOLINE_95_ROOTS | GASOLINE_98_ROOTS

LEVEL3_EXTERNAL_HOLDOUT_ROOTS = {
    "T1", "T4", "T7", "T10",
    "T2", "T5", "T8", "T11",
}
GASOLINE_EXTERNAL_ROOTS = GASOLINE_ROOTS.intersection(LEVEL3_EXTERNAL_HOLDOUT_ROOTS)


def root_from_id(sample_id: str) -> str:
    return str(sample_id).strip().split("-", 1)[0]


def _sort_spectral_columns(df: pd.DataFrame) -> pd.DataFrame:
    wns = df.columns.to_numpy(dtype=float)
    order = np.argsort(wns)
    return df.iloc[:, order]


def label_from_root(root: str) -> str:
    if root in GASOLINE_95_ROOTS:
        return "95"
    if root in GASOLINE_98_ROOTS:
        return "98"
    raise RuntimeError(f"Unknown gasoline root: {root}")


def load_positive_data(paths: Paths) -> Tuple[np.ndarray, pd.DataFrame]:
    df = pd.read_excel(paths.data, sheet_name="Positive", index_col=0)
    df = df.sort_index()
    df = _sort_spectral_columns(df)

    meta = df.index.to_series().to_frame(name="sample_id")
    meta["root"] = meta["sample_id"].apply(root_from_id)
    meta = meta.set_index("sample_id")

    X_raw = df.to_numpy(dtype=float)
    return X_raw, meta


def load_gasoline_training_and_external_from_positive(
    paths: Paths,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X_all, meta = load_positive_data(paths)
    roots_all = meta["root"].to_numpy()

    gasoline_mask = np.isin(roots_all, list(GASOLINE_ROOTS))
    external_mask = np.isin(roots_all, list(GASOLINE_EXTERNAL_ROOTS))
    train_mask = gasoline_mask & (~external_mask)
    ext_mask = gasoline_mask & external_mask

    if not np.any(train_mask):
        raise RuntimeError("No training samples were found.")
    if not np.any(ext_mask):
        raise RuntimeError("No external validation samples were found.")

    X_train_raw = X_all[train_mask]
    y_train = np.array([label_from_root(r) for r in roots_all[train_mask]], dtype=object)

    X_ext_raw = X_all[ext_mask]
    y_ext = np.array([label_from_root(r) for r in roots_all[ext_mask]], dtype=object)

    return X_train_raw, y_train, X_ext_raw, y_ext


def preprocess_snv(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, ddof=1, keepdims=True)
    std[std == 0] = 1.0
    return (X - mean) / std


def fit_snv_pca_lda(
    X_train_raw: np.ndarray,
    y_train: np.ndarray,
    n_components: int = 4,
) -> Tuple[StandardScaler, PCA, LinearDiscriminantAnalysis]:
    X_train_snv = preprocess_snv(X_train_raw)

    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_scaled = scaler.fit_transform(X_train_snv)

    pca = PCA(n_components=n_components)
    scores_train = pca.fit_transform(X_train_scaled)

    lda = LinearDiscriminantAnalysis()
    lda.fit(scores_train, y_train)

    return scaler, pca, lda


def stable_binary_log_posteriors_from_scores(
    lda: LinearDiscriminantAnalysis,
    scores: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    classes = np.asarray(lda.classes_, dtype=object)
    if classes.shape[0] != 2:
        raise ValueError("This helper only supports binary classification.")

    d = np.asarray(lda.decision_function(scores), dtype=float).reshape(-1)

    pred_from_d = np.where(d >= 0.0, classes[1], classes[0])
    pred_model = np.asarray(lda.predict(scores), dtype=object)

    if np.mean(pred_from_d == pred_model) < 0.5:
        d = -d
        pred_from_d = np.where(d >= 0.0, classes[1], classes[0])

    if not np.all(pred_from_d == pred_model):
        raise RuntimeError("decision_function sign could not be matched reliably to lda.predict().")

    log_post_class1 = -np.logaddexp(0.0, -d)
    log_post_class0 = -np.logaddexp(0.0, d)

    return classes, log_post_class0, log_post_class1


def compute_external_outputs(
    scaler: StandardScaler,
    pca: PCA,
    lda: LinearDiscriminantAnalysis,
    X_ext_raw: np.ndarray,
    y_true: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    X_ext_snv = preprocess_snv(X_ext_raw)
    X_ext_scaled = scaler.transform(X_ext_snv)
    X_scores = pca.transform(X_ext_scaled)

    classes, lp0, lp1 = stable_binary_log_posteriors_from_scores(lda, X_scores)

    idx95 = int(np.where(classes == "95")[0][0])
    if idx95 == 0:
        log_post_95 = lp0
        log_post_98 = lp1
    else:
        log_post_95 = lp1
        log_post_98 = lp0

    pred = np.where(log_post_95 >= log_post_98, "95", "98").astype(object)

    top = np.maximum(log_post_95, log_post_98)
    second = np.minimum(log_post_95, log_post_98)
    log10_lr_top_vs_second = (top - second) / np.log(10.0)

    df = pd.DataFrame(
        {
            "is_correct": (y_true == pred),
            "log10_lr_top_vs_second": log10_lr_top_vs_second,
        }
    )

    return log_post_95, log_post_98, df


def summarise_log_lr(
    df_log_post: pd.DataFrame,
    dataset_tag: str,
) -> pd.DataFrame:
    correct_mask = df_log_post["is_correct"].to_numpy(dtype=bool)
    log_lr = df_log_post["log10_lr_top_vs_second"].to_numpy(dtype=float)
    n_samples = int(df_log_post.shape[0])
    n_correct = int(np.count_nonzero(correct_mask))
    accuracy = float(n_correct / n_samples) if n_samples > 0 else np.nan

    if n_correct > 0:
        correct_log_lr = log_lr[correct_mask]
        mean_log_lr = float(np.mean(correct_log_lr))
        min_log_lr = float(np.min(correct_log_lr))
        max_log_lr = float(np.max(correct_log_lr))
        range_text = f"{min_log_lr:.6g} to {max_log_lr:.6g}"
    else:
        mean_log_lr = np.nan
        min_log_lr = np.nan
        max_log_lr = np.nan
        range_text = ""

    return pd.DataFrame(
        [
            {
                "Preprocessing method": "SNV",
                "Dataset": dataset_tag,
                "Number of samples": n_samples,
                "Number of correctly classified samples": n_correct,
                "Classification accuracy": accuracy,
                "Mean log LR (correct classifications)": mean_log_lr,
                "Min log LR (correct classifications)": min_log_lr,
                "Max log LR (correct classifications)": max_log_lr,
                "Associated log(LR) values range (correct classifications)": range_text,
            }
        ]
    )


def plot_lda_log_posterior_external_only_snv(
    log_post_95_ext: np.ndarray,
    log_post_98_ext: np.ndarray,
    y_ext: np.ndarray,
    output_dir: Path,
) -> Path:
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    ax.set_facecolor("#f5f5f5")

    ax.scatter(
        log_post_95_ext[y_ext == "95"],
        log_post_98_ext[y_ext == "95"],
        c="C0",
        s=26,
        alpha=0.78,
        label="#95 Gasoline",
        edgecolors="none",
    )
    ax.scatter(
        log_post_95_ext[y_ext == "98"],
        log_post_98_ext[y_ext == "98"],
        c="C1",
        s=26,
        alpha=0.78,
        label="#98 Gasoline",
        edgecolors="none",
    )

    ax.set_xlabel("Log posterior probability – #95 Gasoline", family="serif")
    ax.set_ylabel("Log posterior probability – #98 Gasoline", family="serif", rotation=270, labelpad=22)
    ax.xaxis.set_label_position("top")
    ax.xaxis.tick_top()
    ax.yaxis.set_label_position("right")
    ax.yaxis.tick_right()
    ax.tick_params(axis="x", which="both", top=True, labeltop=True, bottom=False, labelbottom=False)
    ax.tick_params(axis="y", which="both", right=True, labelright=True, left=False, labelleft=False)

    ax.grid(True, color="#c8c8c8", linewidth=0.6, alpha=0.8)

    ax.spines["top"].set_linewidth(1.1)
    ax.spines["top"].set_color("black")
    ax.spines["right"].set_linewidth(1.1)
    ax.spines["right"].set_color("black")
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)

    ax.tick_params(axis="both", which="both", direction="in", length=5)
    ax.legend(loc="lower left", frameon=True)

    fig.tight_layout()

    out_path = output_dir / "gasoline_95_98_snv_lda_log_posterior_external.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out_path


def run(paths: Paths) -> None:
    paths.output_dir.mkdir(parents=True, exist_ok=True)

    X_train_raw, y_train, X_ext_raw, y_ext = (
        load_gasoline_training_and_external_from_positive(paths)
    )

    scaler, pca, lda = fit_snv_pca_lda(
        X_train_raw=X_train_raw,
        y_train=y_train,
        n_components=4,
    )

    log_post_95_ext, log_post_98_ext, external_log_post_df = compute_external_outputs(
        scaler=scaler,
        pca=pca,
        lda=lda,
        X_ext_raw=X_ext_raw,
        y_true=y_ext,
    )

    plot_path = plot_lda_log_posterior_external_only_snv(
        log_post_95_ext=log_post_95_ext,
        log_post_98_ext=log_post_98_ext,
        y_ext=y_ext,
        output_dir=paths.output_dir,
    )

    log_lr_summary_df = summarise_log_lr(
        df_log_post=external_log_post_df,
        dataset_tag="External validation",
    )

    log_lr_summary_path = (
        paths.output_dir / "gasoline_95_98_snv_external_log_lr_summary.xlsx"
    )
    log_lr_summary_df.to_excel(log_lr_summary_path, index=False)

    print(f"Saved Figure 8: {plot_path}")
    print(f"Saved log(LR) summary: {log_lr_summary_path}")


if __name__ == "__main__":
    cfg = Paths()
    run(cfg)
